### Training a Random Forest Regressor model
Training a random forest regressor that predicts FPL points for the upcoming GW for players. This is a general model, and uses position as one of the predictors. A future step might be to produce a separate model for each position so that position-specific features can be better considered.

Rolling game statistics are key to the model - they will be computed on the previous three games for each player, and used as predictor features.

In [46]:
import os
import torch
import pandas as pd
import numpy as np
from model import AdvancedLSTM
import pickle
from eval import season_performance_with_unlimited_transfers
import json

In [47]:
base_path = os.getcwd()
base_path

'/Users/bragehs/Documents/FPL_forecast/predictor'

In [48]:
data_path = os.path.join(base_path, 'processed_data')
data_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [49]:
X_test = torch.load(data_path + '/X_test.pt', weights_only=True)
y_test = torch.load(data_path + '/y_test.pt', weights_only=True)
test_mapping = pd.read_csv(data_path + '/test_mapping.csv')
player_ids_test = torch.load(data_path + '/test_player_ids.pt', weights_only=True)
pos_ids_test = torch.load(data_path + '/pos_test.pt', weights_only=True)
fixdiff_ids_test = torch.load(data_path + '/fixdiff_test.pt', weights_only=True)

In [50]:
print(X_test.shape, y_test.shape)
print(player_ids_test.shape, pos_ids_test.shape, fixdiff_ids_test.shape)

torch.Size([27283, 5, 25]) torch.Size([27283, 1])
torch.Size([27283]) torch.Size([27283, 5]) torch.Size([27283, 5])


In [56]:
best_model_data = torch.load("best_model_pos_fixdiff_no_xg.pth", map_location=torch.device('cpu'))

/var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/ipykernel_95492/1106457731.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_model_data = torch.load("best_model_pos_

In [57]:
print(best_model_data.keys())
for k, v in best_model_data['model_state_dict'].items():
    if 'embedding' in k:
        print(k, v.shape)

dict_keys(['model_state_dict', 'best_performance', 'hidden_dim', 'num_fc_layers', 'num_layers'])
position_embedding.weight torch.Size([6, 16])
fixdiff_embedding.weight torch.Size([6, 16])


In [58]:
with open(f"{data_path}/vocab/player_name_to_idx.json") as f:
    name_to_idx = json.load(f)
with open(f"{data_path}/vocab/unk_id.txt") as f:
    unk_id = int(f.read())

In [59]:
print(best_model_data['hidden_dim'])
print(best_model_data['num_layers'])
print(best_model_data['num_fc_layers'])

128
1
2


In [60]:
model = AdvancedLSTM(input_dim=X_test.shape[-1], hidden_dim=best_model_data['hidden_dim'],
                            output_dim=1, num_layers=best_model_data['num_layers'],
                            dropout=0.0, num_fc_layers=best_model_data['num_fc_layers'],
                            position_vocab_size=6, position_embed_dim=16,
                            fixture_diff_vocab_size=6, fixture_diff_embed_dim=16,
                            )
model.load_state_dict(best_model_data['model_state_dict'])

<All keys matched successfully>

In [61]:
#print number of parameters in the model
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of parameters in the model: {num_params}")

Number of parameters in the model: 104385


In [62]:
model.eval()

AdvancedLSTM(
  (position_embedding): Embedding(6, 16, padding_idx=0)
  (fixdiff_embedding): Embedding(6, 16, padding_idx=0)
  (lstm): LSTM(57, 128, batch_first=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (relu): ReLU()
  (head): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.0, inplace=False)
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [63]:
test = pd.read_csv(data_path + '/test_data.csv')

In [64]:
features = ['lagged_was_home', 'last_1_assists', 'last_1_bonus', 'last_1_clean_sheets', 'last_1_creativity', 
            'last_1_goals_conceded', 'last_1_goals_scored', 'last_1_ict_index', 'last_1_influence', 'last_1_minutes',
              'last_1_red_cards', 'last_1_threat', 'last_1_yellow_cards', 'last_all_assists', 'last_all_bonus',
                'last_all_clean_sheets', 'last_all_creativity', 'last_all_goals_conceded', 'last_all_goals_scored', 
                'last_all_ict_index', 'last_all_influence', 'last_all_minutes', 
                'last_all_red_cards', 'last_all_threat', 'last_all_yellow_cards']

In [65]:
def forward_model(x, pid=None, pos=None, fixd=None):
    kwargs = {}
    if getattr(model, 'use_player', False) and pid is not None:
        kwargs['player_ids'] = pid
    if getattr(model, 'use_position', False) and pos is not None:
        kwargs['pos_ids'] = pos
    if getattr(model, 'use_fixdiff', False) and fixd is not None:
        kwargs['fixdiff_ids'] = fixd
    try:
        return model(x, **kwargs) if kwargs else model(x)
    except TypeError:
        return model(x)

In [66]:
predictions = forward_model(X_test, pos=pos_ids_test, fixd=fixdiff_ids_test).detach().numpy()
print(predictions.shape)
print(y_test.shape)

(27283, 1)
torch.Size([27283, 1])


In [67]:
def make_predicted_table(y_test, y_pred):
    '''
    Create a DataFrame for LSTM model predictions.
    This needs to keep track of the Gameweek (GW) and player names.
    '''
    test_mapping = pd.read_csv(data_path + '/test_mapping.csv')
    predictions_df = test_mapping.copy()
    predictions_df['actual'] = y_test
    predictions_df['predicted'] = y_pred
    predictions_df['predicted'] = predictions_df['predicted'].where(predictions_df['minutes'] > 0, 0)
    predictions_df.rename(columns={'prediction_gw': 'GW'}, inplace=True)


    # Update the predictions_df reference
    predictions_df = predictions_df.drop_duplicates(subset=['name', 'GW'], keep='last')
    
    return predictions_df


In [68]:
df = make_predicted_table(y_test, predictions)

In [69]:
salah = df[df['name'] == 'mohamed_salah']
salah

,sequence_idx,element,season_x,name,GW,team_x,value,minutes,last_1_goals_scored,last_1_assists,padding_used,position_encoded,actual,predicted
12426,12426,328.0,2024-25,mohamed_salah,1.0,NaN,125.0,90.0,0.00,0.00,4,3.0,14.0,0.898449
12427,12427,328.0,2024-25,mohamed_salah,2.0,NaN,125.0,82.0,0.25,0.25,3,3.0,10.0,4.446022
12428,12428,328.0,2024-25,mohamed_salah,3.0,NaN,126.0,90.0,0.25,0.00,2,3.0,17.0,4.363487
12429,12429,328.0,2024-25,mohamed_salah,4.0,NaN,127.0,90.0,0.25,0.50,1,3.0,2.0,4.718606
12430,12430,328.0,2024-25,mohamed_salah,5.0,NaN,127.0,90.0,0.00,0.00,0,3.0,6.0,4.563841
12431,12431,328.0,2024-25,mohamed_salah,6.0,NaN,128.0,90.0,0.00,0.25,0,3.0,10.0,4.898650
12432,12432,328.0,2024-25,mohamed_salah,7.0,NaN,127.0,72.0,0.25,0.00,0,3.0,3.0,4.825376
12433,12433,328.0,2024-25,mohamed_salah,8.0,NaN,126.0,90.0,0.00,0.00,0,3.0,12.0,4.582381
12434,12434,328.0,2024-25,mohamed_salah,9.0,NaN,126.0,90.0,0.25,0.25,0,3.0,10.0,4.638055
12435,12435,328.0,2024-25,mohamed_salah,10.0,NaN,127.0,90.0,0.25,0.00,0,3.0,9.0,4.761623


In [70]:
print(torch.mean(y_test))
print(torch.var(y_test))    

tensor(1.1469)
tensor(5.3394)


In [71]:
print(np.mean(predictions))
print(np.var(predictions))
print(np.max(predictions))

1.1774632
1.7513747
5.2996902


In [72]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

rmse = root_mean_squared_error(y_test, predictions)
print(f"RMSE: {rmse}")

mae = mean_absolute_error(y_test, predictions)
print(f"MAE: {mae}")


RMSE: 1.9377245903015137
MAE: 1.0418641567230225


In [73]:
X_test.shape

torch.Size([27283, 5, 25])

In [74]:
scores, total_score = season_performance_with_unlimited_transfers(
    y_test=y_test,
    predictions=predictions,
    remaining_lagged_features=features
)

Players with NaN total_points_last_season: []
Number of NaN values remaining: 0
1.0 :  lukasz_fabianski
2.0 :  sepp_van_den_berg
3.0 :  tyler_dibling
4.0 :  daniel_jebbison
Bench players: ['lukasz_fabianski', 'sepp_van_den_berg', 'tyler_dibling', 'daniel_jebbison']
Bench cost: 170.0
Simulating season with unlimited transfers for 38 gameweeks
Available budget per gameweek: 830.0

--- Gameweek 1.0 ---
Players available for GW 1.0: 668
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/83e8f1ea01954ae1b8a7e3bb91966766-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/83e8f1ea01954ae1b8a7e3bb91966766-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 30 COLUMNS
At line 3687 RHS
At line 3713 BOUNDS
At line 

In [75]:
total_score.item()

2192.0

In [76]:
scores

,team,gw_score
0,"[alexis_mac_allister, dejan_kulusevski, domini...",39.0
1,"[bernardo_veiga_de_carvalho_e_silva, bukayo_sa...",64.0
2,"[andrew_robertson, antoine_semenyo, bukayo_sak...",81.0
3,"[andrew_robertson, bukayo_saka, dean_henderson...",48.0
4,"[bukayo_saka, dwight_mcneil, erling_haaland, g...",61.0
5,"[andre_onana, bryan_mbeumo, diogo_dalot_teixei...",60.0
6,"[cole_palmer, cristian_romero, dejan_kulusevsk...",44.0
7,"[andre_onana, brennan_johnson, cole_palmer, er...",35.0
8,"[brennan_johnson, cole_palmer, dwight_mcneil, ...",55.0
9,"[bryan_mbeumo, erling_haaland, jarrod_bowen, j...",48.0
